# Swan · Step 6 · Testing the routine model on catalogue v2

**Imane Moujahid**

I match Loredana’s two v2 CSVs by product code. I check catalogue coverage for France and test ranking and routine assembly for acne, dryness and sensitivity.

To rerun the notebook, I install `requirements-notebooks.txt` and put the three inputs in `raw_data/` at the repository root. I use [Loredana’s two CSVs from Slack](https://lewagon-alumni.slack.com/archives/D0CGCE62DPY/p1794353020215139) and my CosIng extract, `Swan_CosIng_reference_2026-10-26.csv`. I can set `SWAN_DATA_DIR` to use another data folder. I open the notebook from the repository root or `notebooks/` and run all cells.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

workspace = Path.cwd()
if workspace.name == "notebooks":
    workspace = workspace.parent

data_dir = Path(os.environ.get("SWAN_DATA_DIR", workspace / "raw_data"))
catalogue = pd.read_csv(data_dir / "catalogue_v2.csv", dtype="string")
matrix = pd.read_csv(data_dir / "ingredient_matrix_v2.csv", dtype="string")

- I use `Path` to resolve the project and input paths.
- I read barcodes as text to preserve leading zeros. I convert flags where the model uses them.
- I use `display` to show the results as notebook tables.

In [ ]:
required_fields = {
    "code", "role", "product_name", "france", "is_recommendable",
    "catalogue_review_complete", "has_parfum", "has_essential_oil",
    "ingredients_text", "inci_text", "has_acne_active", "is_gentle",
}
missing_fields = required_fields - set(catalogue.columns)
if missing_fields or "code" not in matrix:
    raise ValueError(f"Check the input columns. Missing catalogue columns: {sorted(missing_fields)}.")

for name, table in (("catalogue", catalogue), ("ingredient matrix", matrix)):
    codes = table["code"].str.strip()
    if codes.fillna("").eq("").any() or codes.duplicated().any():
        raise ValueError(f"Missing or duplicate product codes in {name}.")
    table["code"] = codes

if set(catalogue["code"]) != set(matrix["code"]):
    raise ValueError("Both files must cover the same product codes.")

matrix_by_code = matrix.set_index("code").reindex(catalogue["code"])
print(f"{len(catalogue)} products matched, {matrix_by_code.shape[1]} ingredient columns.")

- `required_fields` lists the catalogue columns I use. The notebook reports any missing columns before ranking.
- I trim product codes and reject blank or duplicate codes.
- I check that both files cover the same codes, then align the ingredient matrix to the catalogue order with `reindex`. I can look up ingredients by code. The scoring rules below use catalogue fields and CosIng evidence.

## Reuse the model from Step 5

I reuse the scoring rules, ranking and assembly functions so this notebook can run on its own. `rank_by_role` filters candidates, applies scores and sorts by score and product code. `choose_products` maximizes the number of filled steps before minimizing the sum of ranks. `assemble_routine` returns the selected products and explains each step.

In [ ]:
SCORING_RULES = [
    ("acne_prone", "cleanser", None, 0, "Acne signal by role to be documented"),
    ("acne_prone", "moisturizer", "has_acne_active", 1, "MVP signal, used only if reviewed in the new catalogue"),
    ("acne_prone", "sunscreen", None, 0, "Declared SPF, no provisional acne bonus"),
    ("dryness_barrier", "cleanser", None, 0, "Gentleness criterion to review with Lina"),
    ("dryness_barrier", "moisturizer", "moisturizing_signal", 1, "Moisturizing or humectant function detected"),
    ("dryness_barrier", "sunscreen", "moisturizing_signal", 1, "Bonus after role and SPF verification"),
    ("sensitivity_irritation", "cleanser", None, 0, "Apply the requested exclusions"),
    ("sensitivity_irritation", "moisturizer", None, 0, "Apply the requested exclusions"),
    ("sensitivity_irritation", "sunscreen", None, 0, "Apply exclusions after role verification"),
]

rules = pd.DataFrame(SCORING_RULES, columns=["concern", "role", "signal", "weight", "condition"])
display(rules)

In [ ]:
ROUTINE_ROLES = ("cleanser", "moisturizer", "sunscreen")
EXCLUSION_FIELDS = {"fragrance": "has_parfum", "essential_oils": "has_essential_oil"}


def as_flag(values):
    labels = values.astype("string").str.strip().str.lower()
    return labels.map({"true": True, "false": False}).astype("boolean")


def rank_by_role(products, concern, *, avoid=(), rules=rules):
    if concern not in rules["concern"].unique():
        raise ValueError("Unknown skin concern.")
    if isinstance(avoid, str) or any(item not in EXCLUSION_FIELDS for item in avoid):
        raise ValueError("Unknown exclusion.")

    candidates = products.copy()
    candidates["code"] = candidates["code"].astype("string").str.strip()
    invalid_code = candidates["code"].fillna("").eq("")
    if invalid_code.any() or candidates.duplicated(["role", "code"]).any():
        raise ValueError("The catalogue must have one code per product within each role.")

    results = {}
    for role in ROUTINE_ROLES:
        eligible = candidates.loc[candidates["role"].eq(role)].copy()
        eligible = eligible.loc[as_flag(eligible["role_verified"]).eq(True).fillna(False)]
        if role == "sunscreen":
            eligible = eligible.loc[as_flag(eligible["sun_protection_verified"]).eq(True).fillna(False)]
        for item in avoid:
            field = EXCLUSION_FIELDS[item]
            if field not in eligible:
                eligible = eligible.iloc[0:0]
            else:
                eligible = eligible.loc[as_flag(eligible[field]).eq(False).fillna(False)]

        eligible["score"] = 0
        eligible["applied_signals"] = [[] for _ in range(len(eligible))]
        matching_rules = rules.loc[rules["concern"].eq(concern) & rules["role"].eq(role)]
        for rule in matching_rules.itertuples(index=False):
            if rule.weight and rule.signal in eligible:
                earns_points = as_flag(eligible[rule.signal]).eq(True).fillna(False)
                eligible.loc[earns_points, "score"] += rule.weight
                eligible.loc[earns_points, "applied_signals"] = (
                    eligible.loc[earns_points, "applied_signals"].map(lambda signals: signals + [rule.signal])
                )

        results[role] = eligible.sort_values(["score", "code"], ascending=[False, True])
    return results

In [ ]:
from itertools import product


def choose_products(ranked):
    options = []
    max_choices = len(ROUTINE_ROLES)
    for role in ROUTINE_ROLES:
        rows = ranked[role].head(max_choices).to_dict(orient="records")
        options.append(list(enumerate(rows, start=1)) + [(max_choices + 1, None)])

    best_key = None
    best_selection = None
    for selection in product(*options):
        codes = [row["code"] for _, row in selection if row is not None]
        if len(codes) != len(set(codes)):
            continue

        positions = tuple(position for position, _ in selection)
        rank_sum = sum(position for position, row in selection if row is not None)
        key = (-len(codes), rank_sum, positions)
        if best_key is None or key < best_key:
            best_key = key
            best_selection = selection

    return best_selection

In [ ]:
SIGNAL_LABELS = {
    "moisturizing_signal": "moisturizing or humectant function detected",
    "has_acne_active": "acne signal recorded in catalogue v2",
}


def assemble_routine(ranked):
    selection = choose_products(ranked)
    steps = []
    for role, (position, row) in zip(ROUTINE_ROLES, selection):
        if row is None:
            reason = (
                "No eligible product for this step."
                if ranked[role].empty
                else "The available product is already used in this routine."
            )
            steps.append({"role": role, "code": None, "rank": None, "score": None, "reason": reason})
            continue

        labels = [SIGNAL_LABELS.get(signal, signal) for signal in row["applied_signals"]]
        reason = ", ".join(labels) if labels else "Eligible product with no score bonus for this profile."
        same_score = ranked[role]["score"].eq(row["score"]).sum()
        if position == 1 and same_score > 1:
            reason += f" Product code broke the tie among {same_score} candidates with the same score."
        if position > 1:
            reason += f" Ranked {position} for this step and selected to avoid a duplicate product."
        steps.append({
            "role": role,
            "code": row["code"],
            "rank": position,
            "score": int(row["score"]),
            "reason": reason,
        })

    filled = sum(step["code"] is not None for step in steps)
    status = {0: "no_result", len(ROUTINE_ROLES): "ok"}.get(filled, "partial")
    return {"status": status, "steps": steps}

## Prepare the model inputs

I use the catalogue’s `france` flag to define the France test subset. I record the declared SPF for each of the seven sunscreen candidates where I can link it to the exact product. Retail availability in France still needs verification.

In [ ]:
# I record the declared SPF and keep each source link for review.
spf_review = pd.DataFrame([
    ("3468080009332", "50", "https://www.corinedefarme.fr/p/creme-protectrice-visage-et-corps-spf50/", "https://www.ezam.cz/pece-o-plet-a-pokozku/75210-opalovaci-krem-na-telo-a-oblicej-corine-de-farme-spf-50-50-ml-3468080009332.html", False),
    ("4743318143293", "30", "https://organicshop.co.uk/products/organic-shop-sunscreen-day-face-cream-30-spf-oily-skin-50ml", "https://www.pandashop.md/ru/product/sunscreen-day-face-cream-peach-spf30-50ml-00804499/", False),
    ("4005900458773", "50", "https://www.nivea.fr/produits/cr%C3%A8me-uv-visage-anti-%C3%A2ge-q10-spf-50-40060002197210040.html", "https://www.farmaline.be/fr/beaute/BE09802538/nivea-sun-uv-gezicht-anti-age-spf50.htm", False),
    ("42449010", pd.NA, pd.NA, pd.NA, False),
    ("3282770202120", "50+", "https://www.vidal.fr/parapharmacie/aderma-protect-spf-50-creme-sans-parfum-195613.html", pd.NA, True),
    ("4005900328946", "50+", "https://swiss-tablets.com/202394-nivea-sun-alpin-sun-cream-face-spf-50-50-ml/", pd.NA, False),
    ("3506770501008", "50", "https://www.bol.com/nl/nl/p/lovea-bio-anti-ageing-spf-50-50-ml-zonnebrand-creme/9200000005932746/", pd.NA, False),
], columns=["code", "declared_spf", "spf_source", "code_source", "reported_discontinued"])

reviewed = as_flag(catalogue["catalogue_review_complete"]).fillna(False)
recommended = as_flag(catalogue["is_recommendable"]).fillna(False)
france = as_flag(catalogue["france"]).fillna(False)

products = catalogue.loc[france].copy()
products["role_verified"] = (reviewed & recommended).loc[products.index]
products["sun_protection_verified"] = pd.Series(pd.NA, index=products.index, dtype="boolean")
if "sun_protection_verified" in catalogue:
    products["sun_protection_verified"] = as_flag(catalogue.loc[products.index, "sun_protection_verified"])

In [ ]:
solaires = products.loc[products["role"].eq("sunscreen") & products["role_verified"]]
spf_audit = solaires[["code", "product_name"]].merge(
    spf_review, on="code", how="left", validate="one_to_one"
)
verified_codes = spf_audit.loc[spf_audit["declared_spf"].notna(), "code"]
discontinued_codes = spf_audit.loc[spf_audit["reported_discontinued"].eq(True), "code"]

# I give reviewed flags in a new export priority.
missing_spf = products["sun_protection_verified"].isna()
products.loc[missing_spf & products["code"].isin(verified_codes), "sun_protection_verified"] = True
products = products.loc[~products["code"].isin(discontinued_codes)].copy()
display(spf_audit[["code", "product_name", "declared_spf", "reported_discontinued"]])

In [ ]:
eligible_by_catalogue = catalogue.loc[reviewed & recommended]
display(pd.DataFrame({
    "reviewed_catalogue": eligible_by_catalogue.groupby("role").size(),
    "listed_for_france": catalogue.loc[reviewed & recommended & france].groupby("role").size(),
}).reindex(ROUTINE_ROLES).fillna(0).astype(int))

- `as_flag` converts the CSV flags to true, false or unknown. I require positive France and role-review flags for the corresponding checks.
- I match the SPF review to the catalogue by product code and give an explicit decision in a future export priority. I leave the Nivea SPF 30 reference unknown because I could not verify the exact code.
- I exclude the A-Derma reference from ranking because Vidal reports that it was discontinued. I keep its declared SPF in the review table.
- I show the reviewed catalogue counts by role and the counts for the France subset.

## Build scoring signals and check exclusions

I reuse the CosIng calculation from Step 5 for dryness. Loredana’s notebook shows that a negative fragrance or essential oil flag means the script found no match in the normalized text. I treat unverified absences as unknown.

In [ ]:
import re
import unicodedata

ALIASES = {
    "WATER": "AQUA",
    "EAU": "AQUA",
    "AQUA/WATER/EAU": "AQUA",
    "AQUA/WATER": "AQUA",
    "WATER/AQUA": "AQUA",
    "WATER (AQUA)": "AQUA",
    "AQUA (WATER)": "AQUA",
    "AQUA (WATER/EAU)": "AQUA",
    "WATER/AQUA/EAU": "AQUA",
    "AQUA (WATER, EAU)": "AQUA",
    "FRAGRANCE": "PARFUM",
    "FRAGRANCE (PARFUM)": "PARFUM",
    "PARFUM (FRAGRANCE)": "PARFUM",
}
FRAGRANCE_MARKERS = {"PARFUM", "PERFUME", "AROMA"}
SEPARATORS = re.compile(r"(?<!\d),|,(?!\d)|[;\n•]")


def inci_key(name):
    normalized = unicodedata.normalize("NFKC", name)
    return " ".join(normalized.upper().strip().strip("•·").split())


def ingredient_keys(text):
    if pd.isna(text):
        return []
    text = re.sub(r"^\s*(?:ingredients|ingrédients|inci)\s*:\s*", "", text, flags=re.I)
    parts = []
    depth = 0
    start = 0
    for index, char in enumerate(text):
        if char == "(":
            depth += 1
        elif char == ")":
            depth = max(0, depth - 1)
        elif depth == 0 and SEPARATORS.match(text, index):
            parts.append(text[start:index])
            start = index + 1
    parts.append(text[start:])

    keys = [inci_key(part) for part in parts]
    return [ALIASES.get(key, key) for key in keys if key]

In [ ]:
cosing = pd.read_csv(data_dir / "Swan_CosIng_reference_2026-10-26.csv", dtype="string")
reference = cosing.assign(key=cosing["inci_name"].dropna().map(inci_key))
unique_names = reference.loc[reference["key"].notna() & ~reference["key"].duplicated(keep=False)]
functions_by_name = {
    row.key: {part.strip().upper() for part in row.functions.split("|") if part.strip()}
    for row in unique_names[["key", "functions"]].dropna().itertuples(index=False)
}
FUNCTION_GROUPS = {
    "moisturizing_signal": {"MOISTURISING", "HUMECTANT", "SKIN CONDITIONING - HUMECTANT"},
    "detected_uv_filter": {"UV FILTER"},
}


def detect_evidence(keys):
    found = {
        signal: sorted({key for key in keys if functions_by_name.get(key, set()) & functions})
        for signal, functions in FUNCTION_GROUPS.items()
    }
    found["detected_fragrance"] = sorted(set(keys) & FRAGRANCE_MARKERS)
    return found



evidence = products["ingredients_text"].map(ingredient_keys).map(detect_evidence)
products["moisturizing_signal"] = evidence.map(lambda row: bool(row["moisturizing_signal"]))
products["has_acne_active"] = as_flag(products["has_acne_active"])
signals = pd.DataFrame(index=catalogue.index)
for field in EXCLUSION_FIELDS.values():
    signals[field + "_source"] = as_flag(catalogue[field])
    signals[field] = signals[field + "_source"].where(signals[field + "_source"].eq(True))

# I also check for fragrance terms followed by punctuation or a reference.
fragrance_pattern = r"\b(?:PARFUM|FRAGRANCE|PERFUME|AROMA)\b"
fragrance_in_raw = catalogue["ingredients_text"].str.contains(
    fragrance_pattern, case=False, regex=True, na=False
)
signals.loc[fragrance_in_raw, "has_parfum"] = True
signals["is_gentle_source"] = as_flag(catalogue["is_gentle"])
signals["is_gentle"] = signals["is_gentle_source"].mask(signals["has_parfum"].eq(True).fillna(False), False)
products = products.drop(columns=[*EXCLUSION_FIELDS.values(), "is_gentle"]).join(signals)

disagreements = fragrance_in_raw & signals["has_parfum_source"].eq(False).fillna(False)
display(catalogue.loc[disagreements & reviewed & recommended, ["code", "product_name", "france"]])

- `ingredient_keys` splits the INCI list, handles parentheses and normalizes names. `detect_evidence` finds ingredients with a moisturizing or humectant function in CosIng.
- I use the CSV’s acne flag to test the provisional scoring rule. I preserve the source `is_gentle` value and disable the flag where fragrance is detected. The current sensitivity ranking awards no gentleness points.
- I keep the original exclusion flags with a `_source` suffix. `where` preserves reported presences and leaves other values unknown.
- I search the raw ingredients for fragrance terms, including when a term is followed by punctuation or a reference. I report disagreements with the catalogue flag. An unmatched search leaves the status unknown.

## Check the Bioderma regression

I test this product separately because the catalogue places it outside the France subset. I verify that it passes the role check, is excluded when fragrance is selected and has its gentleness flag disabled.

In [ ]:
bioderma_code = "3401395376935"
sample = catalogue.loc[catalogue["code"].eq(bioderma_code), ["code", "role"]].join(signals)
if not sample.empty:
    sample["role_verified"] = (reviewed & recommended).loc[sample.index]
    sample["sun_protection_verified"] = pd.NA
    row = sample.iloc[0]
    assert row["has_parfum"] == True
    assert row["is_gentle"] == False
    concern = "sensitivity_irritation"
    assert bioderma_code in rank_by_role(sample, concern)["cleanser"]["code"].tolist()
    assert bioderma_code not in rank_by_role(sample, concern, avoid=("fragrance",))["cleanser"]["code"].tolist()
    print("Bioderma regression passed. I detected fragrance, applied the exclusion and disabled the gentleness flag.")

## Run the scenarios

For each concern, I run the model without exclusions, with each exclusion on its own and with both together. I compare candidate counts and routine outcomes. For the scenarios without exclusions, I show the products, the reasons for each pick and the number of candidates at the same score.

In [ ]:
summary_rows = []
routine_rows = []
shuffled_products = products.sample(frac=1, random_state=42)
for concern in rules["concern"].unique():
    for avoid in ((), ("fragrance",), ("essential_oils",), ("fragrance", "essential_oils")):
        ranked = rank_by_role(products, concern, avoid=avoid)
        routine = assemble_routine(ranked)
        assert routine == assemble_routine(rank_by_role(shuffled_products, concern, avoid=avoid))
        selected = [step["code"] for step in routine["steps"] if step["code"] is not None]
        assert len(selected) == len(set(selected))
        assert ranked["sunscreen"]["sun_protection_verified"].eq(True).fillna(False).all()
        assert not ranked["sunscreen"]["code"].isin(discontinued_codes).any()
        for item in avoid:
            field = EXCLUSION_FIELDS[item]
            assert all(table[field].eq(False).fillna(False).all() for table in ranked.values())

        summary_rows.append({
            "concern": concern,
            "exclusion": ", ".join(avoid) or "none",
            **{role: len(table) for role, table in ranked.items()},
            "routine": routine["status"],
        })
        if not avoid:
            for step in routine["steps"]:
                same_score = ranked[step["role"]]["score"].eq(step["score"]).sum()
                routine_rows.append({"concern": concern, **step, "tied candidates": same_score})

display(pd.DataFrame(summary_rows).set_index(["concern", "exclusion"]))
routine_details = pd.DataFrame(routine_rows).merge(
    products[["code", "product_name"]], on="code", how="left", validate="many_to_one"
)
with pd.option_context("display.max_columns", None):
    display(routine_details.set_index(["concern", "role"])[
        ["product_name", "rank", "score", "tied candidates", "reason"]
    ])

- I run each concern and exclusion combination on the same catalogue.
- I shuffle the input to check that row order leaves the routine identical. I also check product uniqueness, sunscreen SPF evidence, discontinued products and confirmation for each requested exclusion.
- The first table shows coverage and routine status. The second shows each selected product, rank, score and reason.

Product code breaks ties when candidates have the same score. I want to review with Lina why an anti-aging moisturizer ranks first for acne and which signals could improve the sensitivity ranking, where all candidates currently score zero. I also want to agree with the team on the gentleness criteria and the evidence we need to confirm an ingredient’s absence.